In [1]:
# ============================================================
# US_MONTHLY.CSV - NLP + K-MEANS CLUSTERING
# Complete Jupyter Notebook Code
# ============================================================

# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

# Run this cell if the libraries are not already installed.
# Uncomment the line below if required.

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import warnings

warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score


# ============================================================
# 3. DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# ============================================================
# 4. LOAD DATASET
# ============================================================

FILE_NAME = "us_monthly.csv"

df = pd.read_csv(FILE_NAME)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 5. BASIC DATASET INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# 6. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("\nDataset after removing duplicates:")
print(df.shape)


# ============================================================
# 7. FIND TEXT COLUMNS
# ============================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("\nPossible text columns:")
print(text_columns)


# ============================================================
# 8. SELECT TEXT COLUMN
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
# If you know the text column, enter its name here.
#
# Example:
# TEXT_COLUMN = "description"
# TEXT_COLUMN = "text"
# TEXT_COLUMN = "article"
# ------------------------------------------------------------

TEXT_COLUMN = None


# ============================================================
# 9. AUTOMATIC TEXT COLUMN DETECTION
# ============================================================

if TEXT_COLUMN is None:

    if len(text_columns) == 0:
        raise ValueError(
            "No text column was found in the dataset. "
            "K-Means + NLP requires a text column."
        )

    # Calculate average text length for each object column
    text_scores = {}

    for col in text_columns:

        values = df[col].dropna().astype(str)

        if len(values) > 0:
            average_length = values.str.len().mean()
            unique_ratio = values.nunique() / len(values)

            # Prefer columns with reasonably long text
            text_scores[col] = average_length * unique_ratio

    TEXT_COLUMN = max(
        text_scores,
        key=text_scores.get
    )

print("\nSelected text column:")
print(TEXT_COLUMN)


# ============================================================
# 10. DISPLAY SELECTED TEXT
# ============================================================

print("\nExamples from selected text column:")

display(
    df[[TEXT_COLUMN]].head(10)
)


# ============================================================
# 11. HANDLE MISSING TEXT
# ============================================================

df = df.dropna(
    subset=[TEXT_COLUMN]
).copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

print("\nRows after removing missing text:")
print(len(df))


# ============================================================
# 12. INITIALIZE NLP TOOLS
# ============================================================

stop_words = set(
    stopwords.words("english")
)

lemmatizer = WordNetLemmatizer()


# ============================================================
# 13. NLP PREPROCESSING FUNCTION
# ============================================================

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep alphabetic characters only
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 2
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


# ============================================================
# 14. APPLY NLP PREPROCESSING
# ============================================================

df["clean_text"] = df[TEXT_COLUMN].apply(
    preprocess_text
)

print("\nOriginal vs cleaned text:")

display(
    df[
        [TEXT_COLUMN, "clean_text"]
    ].head(10)
)


# ============================================================
# 15. REMOVE EMPTY TEXT
# ============================================================

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

print("\nRows after NLP cleaning:")
print(len(df))


# ============================================================
# 16. TF-IDF VECTORIZATION
# ============================================================

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = tfidf.fit_transform(
    df["clean_text"]
)

print("\n" + "=" * 60)
print("TF-IDF")
print("=" * 60)

print("TF-IDF matrix shape:", X.shape)

print(
    "Number of vocabulary terms:",
    len(tfidf.get_feature_names_out())
)


# ============================================================
# 17. GET VOCABULARY
# ============================================================

terms = tfidf.get_feature_names_out()

print("\nFirst 50 TF-IDF terms:")

print(
    terms[:50]
)


# ============================================================
# 18. FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

# We test K from 2 to 10.

max_k = min(
    10,
    X.shape[0] - 1
)

K_range = range(2, max_k + 1)

inertia = []
silhouette_scores = []


for k in K_range:

    kmeans_temp = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels_temp = kmeans_temp.fit_predict(X)

    inertia.append(
        kmeans_temp.inertia_
    )

    silhouette_scores.append(
        silhouette_score(
            X,
            labels_temp
        )
    )


# ============================================================
# 19. ELBOW METHOD
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(K_range),
    inertia,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for Optimal K"
)

plt.xticks(
    list(K_range)
)

plt.grid(True)

plt.show()


# ============================================================
# 20. SILHOUETTE SCORE
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(K_range),
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(K_range)
)

plt.grid(True)

plt.show()


# ============================================================
# 21. SELECT BEST K
# ============================================================

best_k = list(K_range)[
    np.argmax(silhouette_scores)
]

best_score = max(
    silhouette_scores
)

print("\n" + "=" * 60)
print("OPTIMAL NUMBER OF CLUSTERS")
print("=" * 60)

print("Best K:", best_k)
print(
    "Best Silhouette Score:",
    round(best_score, 4)
)


# ============================================================
# 22. FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)


# ============================================================
# 23. DISPLAY CLUSTER RESULTS
# ============================================================

print("\n" + "=" * 60)
print("CLUSTER RESULTS")
print("=" * 60)

display(
    df[
        [TEXT_COLUMN, "cluster"]
    ].head(20)
)


# ============================================================
# 24. COUNT RECORDS IN EACH CLUSTER
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nRecords in each cluster:")

display(
    cluster_counts.to_frame(
        name="Number of Records"
    )
)


# ============================================================
# 25. CLUSTER SIZE VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "Number of Records in Each Cluster"
)

plt.show()


# ============================================================
# 26. FIND TOP WORDS FOR EACH CLUSTER
# ============================================================

print("\n" + "=" * 60)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 60)

order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)


cluster_keywords = {}


for cluster_number in range(best_k):

    top_indices = (
        order_centroids[
            cluster_number,
            :20
        ]
    )

    top_words = [
        terms[index]
        for index in top_indices
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 27. CREATE CLUSTER SUMMARY TABLE
# ============================================================

cluster_summary = []

for cluster_number in range(best_k):

    cluster_summary.append({
        "Cluster": cluster_number,
        "Number_of_Records": int(
            cluster_counts.get(
                cluster_number,
                0
            )
        ),
        "Top_Words": ", ".join(
            cluster_keywords[
                cluster_number
            ][:10]
        )
    })


cluster_summary_df = pd.DataFrame(
    cluster_summary
)

print("\nCluster Summary:")

display(
    cluster_summary_df
)


# ============================================================
# 28. DISPLAY SAMPLE RECORDS FROM EACH CLUSTER
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE RECORDS FROM EACH CLUSTER")
print("=" * 60)


for cluster_number in range(best_k):

    print(
        f"\n{'=' * 20} "
        f"CLUSTER {cluster_number} "
        f"{'=' * 20}"
    )

    sample = (
        df[
            df["cluster"] == cluster_number
        ][
            [TEXT_COLUMN, "clean_text"]
        ]
        .head(5)
    )

    display(sample)


# ============================================================
# 29. REDUCE TF-IDF DIMENSIONS FOR VISUALIZATION
# ============================================================

# TruncatedSVD is better suited than regular PCA
# for sparse TF-IDF matrices.

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)


# ============================================================
# 30. CREATE VISUALIZATION DATAFRAME
# ============================================================

plot_df = pd.DataFrame({
    "Component_1": X_2d[:, 0],
    "Component_2": X_2d[:, 1],
    "Cluster": df["cluster"].values
})


# ============================================================
# 31. PLOT K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=plot_df,
    x="Component_1",
    y="Component_2",
    hue="Cluster",
    palette="tab10",
    s=70,
    alpha=0.8
)

plt.title(
    "K-Means Clustering using NLP + TF-IDF"
)

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.show()


# ============================================================
# 32. CALCULATE FINAL SILHOUETTE SCORE
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print("\n" + "=" * 60)
print("FINAL MODEL PERFORMANCE")
print("=" * 60)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette Score:",
    round(final_silhouette, 4)
)


# ============================================================
# 33. ADD CLUSTER NAMES
# ============================================================

# Give clusters readable names based on their
# most important words.

cluster_names = {}

for cluster_number in range(best_k):

    words = cluster_keywords[
        cluster_number
    ][:3]

    cluster_names[
        cluster_number
    ] = (
        "Cluster "
        + str(cluster_number)
        + " - "
        + ", ".join(words)
    )


df["cluster_name"] = df[
    "cluster"
].map(cluster_names)


# ============================================================
# 34. DISPLAY FINAL DATASET
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATASET")
print("=" * 60)

display(
    df.head(20)
)


# ============================================================
# 35. SAVE CLUSTERED DATASET
# ============================================================

OUTPUT_FILE = (
    "us_monthly_kmeans_clusters.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "\nClustered dataset saved as:",
    OUTPUT_FILE
)


# ============================================================
# 36. SAVE CLUSTER SUMMARY
# ============================================================

SUMMARY_FILE = (
    "us_monthly_cluster_summary.csv"
)

cluster_summary_df.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    "Cluster summary saved as:",
    SUMMARY_FILE
)


# ============================================================
# 37. FINAL REPORT
# ============================================================

print("\n")
print("=" * 60)
print("FINAL REPORT")
print("=" * 60)

print(
    f"Original dataset rows: "
    f"{df.shape[0]}"
)

print(
    f"Text column used: "
    f"{TEXT_COLUMN}"
)

print(
    f"TF-IDF features: "
    f"{X.shape[1]}"
)

print(
    f"Number of clusters: "
    f"{best_k}"
)

print(
    f"Silhouette score: "
    f"{final_silhouette:.4f}"
)

print("\nCluster sizes:")

for cluster_number in range(best_k):

    count = cluster_counts.get(
        cluster_number,
        0
    )

    print(
        f"Cluster {cluster_number}: "
        f"{count} records"
    )

print("\nTop keywords:")

for cluster_number in range(best_k):

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(
            cluster_keywords[
                cluster_number
            ][:10]
        )
    )

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

DATASET LOADED
Rows: 242
Columns: 2

Column names:
['date', 'y']

First 5 rows:


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,date,y
0,2001-01-01,2676998
1,2001-02-01,2309464
2,2001-03-01,2246633
3,2001-04-01,1807170
4,2001-05-01,1522382



DATASET INFORMATION

Data types:
date      str
y       int64
dtype: object

Missing values:


date    0
y       0
dtype: int64

...

,date
0,2001-01-01
1,2001-02-01
2,2001-03-01
3,2001-04-01
4,2001-05-01
5,2001-06-01
6,2001-07-01
7,2001-08-01
8,2001-09-01
9,2001-10-01



Rows after removing missing text:
242

Original vs cleaned text:


,date,clean_text
0,2001-01-01,
1,2001-02-01,
2,2001-03-01,
3,2001-04-01,
4,2001-05-01,
5,2001-06-01,
6,2001-07-01,
7,2001-08-01,
8,2001-09-01,
9,2001-10-01,



Rows after NLP cleaning:
0


ValueError: empty vocabulary; perhaps the documents only contain stop words